In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:
import torch
from DatasetsModels import PhonemeEmbeddingDataset, ClassificationModel, collate_fn, ClassificationModelCTChead
from GetData import load_selected_embeddings
from ToIPA import corpres_to_ipa_symbol, panphon_features
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from VizulalisationTools import chosen_phonemes_stats
from main import train, test, evaluate
import random
from SaveTools import save_experiment_info
import os
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np



In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA runtime version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version() if torch.cuda.is_available() else 'N/A'}")

# Проверь установку PyTorch
print(f"torch.cuda.is_built(): {torch.backends.cudnn.enabled}")



In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
root_dir = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging'
save_path = str(PROJECT_ROOT)
speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna', 'Galina', 'Victoria', 'Petr', 'Alexander']
phonetic_features = ['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid', 'voi', 'sg', 'cg', 'ant', 'cor', 'distr', 'lab', 'hi', 'lo', 'back', 'round', 'velaric', 'tense', 'long']
phoneme_list_full = ['a0', 'a1', 'a2', 'a4', 'b', "b'", 'c', 'ch', 'ch_', 'd', "d'", 'e0', 'e1', 'e2', 'e4', 'f', "f'", 'g', "g'", 'h', "h'", 'i0', 'i1', 'i2', 'i4', 'j', 'jr', 'jl', 'ji4', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'o1', 'o2', 'o4', 'p', "p'", 'r', "r'", 's', "s'", 'sc', 'sh', 't', "t'", 'u0', 'u1', 'u2', 'u4', 'v', "v'", 'y0', 'y1', 'y2', 'y4', 'z', "z'", 'zh', "zh'", 'C', 'CH', 'H', 'SC']

In [ ]:
debugging = False
save_exp = False
training = True
#model_id = 'VACXUXVEXO'
training_mode = 'triplet' # ['triplet', 'ctc', 'averaged']
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'a1', 'i1','u1', 'y1', ]
# phoneme_list = ['a0', 'a4', 'b', 'd','e0', 'f',  'g', 'h', 'i0','i4', 'j', 'k','l', 'm',  'n', 'o0', 'p', 'r', 's', 'sh', 't', 'u0', 'v',  'y0', 'z', 'zh', 'sil']
# phoneme_list = ['a0', 't', "i0", "t'", "k", 's', "s'" ]  
# интересующие фонемы (без позиций)
samples_of_ph = 1000              # сколько примеров на фонему
train_speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna']
test_speakers = ['Galina', 'Victoria', 'Petr', 'Alexander']

if debugging:
    phoneme_list = ['a0', 't', "i0" ]  
    samples_of_ph = 1            
    train_speakers = ['Vladimir']
    test_speakers = ['Galina']
    

In [ ]:
int_code = [random.randint(65, 90) for i in range(10)]
uid = ''.join([chr(code) for code in int_code])

In [ ]:
log_path = save_path + '\\runs\\' + uid

In [ ]:
num_classes = len(phoneme_list)

In [ ]:
learning_rate = 1e-3
weight_decay=1e-4
batch_size = 32
num_epochs = 20
dropout = 0.1
input_size = 512

In [ ]:
save_model_path = r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\phonetic_features_predictions'

In [ ]:
train_embeddings, train_phonemes_nobord, _ = load_selected_embeddings(root_dir, phoneme_list, samples_of_ph, train_speakers,  triplet_variation='no borders')

test_embeddings, test_phonemes_nobord, _ = load_selected_embeddings(root_dir, phoneme_list, samples_of_ph, test_speakers,  triplet_variation='no borders')


In [ ]:
# class PhonemeEmbeddingDataset(Dataset):
# 
#     def __init__(self, data, phoneme_list, corpres_to_ipa_symbol=corpres_to_ipa_symbol, panphon_features=panphon_features, triplet_labels=False):
#         self.phoneme_list = phoneme_list
#         self.data = data
#         self.phoneme2idx = {ph: i for i, ph in enumerate(self.phoneme_list)}
#         self.num_phonemes = len(self.phoneme2idx)
#         self.triplet_labels = triplet_labels
# 
#         # Функция для перевода в IPA
#         self.corpres_to_ipa_symbol = corpres_to_ipa_symbol
#         self.panphon_features = panphon_features
# 
#         # Преобразуем метки в индексы и в IPA
#         self.labels = []          # индексы по self.phoneme_list
#         self.labels_ipa = []      # строки IPA для каждой позиции
#         self.features_ipa = [] 
#         self.features_ipa_adapted = []
#         def encode_features(vector):
#             return [v + 1 for v in vector]
# 
#         for d in data:
#             label = []
#             label_ipa = []
#             feature_ipa = []
#             feature_ipa_adapted =[]
#             for ph in d['label']:
#                 if ph in self.phoneme2idx:
#                     ph_idx = self.phoneme2idx[ph]
#                 else:
#                     ph_idx = -1
# 
#                 label.append(ph_idx)
# 
#                 # Конвертируем в IPA, если функция задана
#                 if self.corpres_to_ipa_symbol is not None:
#                     ipa_ph = self.corpres_to_ipa_symbol(ph)
#                     ph_feats = self.panphon_features(ipa_ph)
#                     ph_feats_adapted = encode_features(ph_feats[0].numeric())
# 
#                 else:
#                     ipa_ph = ph
#                     ph_feats = None
#                     ph_feats_adapted = None
#                 label_ipa.append(ipa_ph)
#                 feature_ipa.append(ph_feats)
#                 feature_ipa_adapted.append(ph_feats_adapted)
# 
#             self.labels.append(label)
#             self.labels_ipa.append(label_ipa)
#             self.features_ipa.append(feature_ipa)
#             self.features_ipa_adapted.append(feature_ipa_adapted)
# 
#     def __len__(self):
#         return len(self.data)
# 
#     def __getitem__(self, idx):
#         if self.triplet_labels:
#             
#             embedding = self.data[idx]['embedding']          # Tensor
#             phonemes_original = self.data[idx]['label']     # список исходных символов
#             ph_idx_list = self.labels[idx]                   # список индексов
#             phonemes_ipa = self.labels_ipa[idx]              # список IPA‑символов
#             phonetic_features = self.features_ipa[idx]
#             phonemic_features_adapted = self.features_ipa_adapted[idx]
#     
#             phoneme_labels = torch.tensor(ph_idx_list, dtype=torch.long)
#         else:
#             embedding = self.data[idx]['embedding']          # Tensor
#             phonemes_original = self.data[idx]['label'][1]     # список исходных символов
#             ph_idx_list = self.labels[idx][1]                   # список индексов
#             phonemes_ipa = self.labels_ipa[idx][1]             # список IPA‑символов
#             phonetic_features = self.features_ipa[idx][1]
#             phonemic_features_adapted = self.features_ipa_adapted[idx][1]
#     
#             phoneme_labels = torch.tensor(ph_idx_list, dtype=torch.long)
#         
#         
#         
# 
#         return embedding, phoneme_labels, phonemes_original, phonemes_ipa, torch.tensor(phonemic_features_adapted, dtype=torch.long)


In [ ]:
class PhonemeEmbeddingDataset(Dataset):
    def __init__(
        self,
        data,
        phoneme_list,
        corpres_to_ipa_symbol=None,
        panphon_features=None,
        triplet_labels=False,
        target_position=1,   # какую позицию использовать если НЕ триплет
    ):
        super().__init__()

        self.data = data
        self.triplet_labels = triplet_labels
        self.target_position = target_position

        self.phoneme_list = phoneme_list
        self.phoneme2idx = {ph: i for i, ph in enumerate(self.phoneme_list)}
        self.num_phonemes = len(self.phoneme2idx)

        self.corpres_to_ipa_symbol = corpres_to_ipa_symbol
        self.panphon_features = panphon_features

        # Кэш признаков по фонеме
        self.phoneme_feature_cache = {}

        if self.corpres_to_ipa_symbol is not None and self.panphon_features is not None:
            self._precompute_features()

    def _precompute_features(self):

        def encode_features(vector):
            result_vector = []
            for v in vector:
                if v == -1: #no feature
                    result_vector.append(0)
                if v == 1: # feature present
                    result_vector.append(1)
                if v == 0: # not applied
                    result_vector.append(2)
                    
            return result_vector

        for ph in self.phoneme_list:

            ipa_ph = self.corpres_to_ipa_symbol(ph)
            ph_feats = self.panphon_features(ipa_ph)

            if ph_feats and len(ph_feats) > 0:
                feats_numeric = ph_feats[0].numeric()
                feats_adapted = encode_features(feats_numeric)
            else:
                feats_adapted = None

            self.phoneme_feature_cache[ph] = {
                "ipa": ipa_ph,
                "features": feats_adapted
            }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]
        embedding = sample["embedding"]

        if self.triplet_labels:
            phonemes = sample["label"]
        else:
            phonemes = [sample["label"][self.target_position]]

        ph_indices = []
        ipa_symbols = []
        phonetic_features = []

        for ph in phonemes:

            # индекс
            ph_idx = self.phoneme2idx.get(ph, -1)
            ph_indices.append(ph_idx)

            # признаки (если есть)
            if ph in self.phoneme_feature_cache:
                ipa_symbols.append(self.phoneme_feature_cache[ph]["ipa"])
                phonetic_features.append(self.phoneme_feature_cache[ph]["features"])
            else:
                ipa_symbols.append(ph)
                phonetic_features.append(None)

        phoneme_labels = torch.tensor(ph_indices, dtype=torch.long)

        if phonetic_features[0] is not None:
            phonetic_features_tensor = torch.tensor(
                phonetic_features,
                dtype=torch.long
            )
        else:
            phonetic_features_tensor = None

        return (
            embedding,
            phoneme_labels,
            phonemes,
            ipa_symbols,
            phonetic_features_tensor[0]
        )

In [ ]:
train_dataset = PhonemeEmbeddingDataset(train_embeddings, phoneme_list, corpres_to_ipa_symbol=corpres_to_ipa_symbol, panphon_features=panphon_features)
test_dataset = PhonemeEmbeddingDataset(test_embeddings, phoneme_list, corpres_to_ipa_symbol=corpres_to_ipa_symbol, panphon_features=panphon_features)

In [ ]:
train_dataset[-1]

In [ ]:
label_counter = Counter()
phoneme_counter = Counter()
ipa_counter = Counter()
feature_sum = None
feature_count = 0

for i in range(len(train_dataset)):
    _, labels, phonemes, ipa_symbols, features = train_dataset[i]

    # если не триплет - приводим всё к списку
    if not train_dataset.triplet_labels:
        labels = [labels.item()]
        phonemes = [phonemes[0]]
        ipa_symbols = [ipa_symbols[0]]
        features = [features] if features is not None else []

    # считаем частоты
    for l in labels:
        label_counter[int(l)] += 1
    for ph in phonemes:
        phoneme_counter[ph] += 1
    for ipa in ipa_symbols:
        ipa_counter[ipa] += 1        

In [ ]:
plt.figure()
plt.bar(phoneme_counter.keys(), phoneme_counter.values())
plt.xticks(rotation=90)
plt.title("Phoneme Distribution")
plt.show()

In [ ]:
plt.figure()
plt.bar(ipa_counter.keys(), ipa_counter.values())
plt.xticks(rotation=90)
plt.title("IPA Distribution")
plt.show()

In [ ]:
# будем считать: axis=0 — признаки, axis=1 — значение (0,1,2)
feature_counts = None

for i in range(len(train_dataset)):
    _, labels, phonemes, ipa_symbols, features = train_dataset[i]

    if not train_dataset.triplet_labels:
        labels = [labels.item()]
        phonemes = [phonemes[0]]
        ipa_symbols = [ipa_symbols[0]]
        features = [features] if features is not None else []

    for f in features:
        f = f.numpy().astype(int)  # предполагаем значения 0/1/2, форма (num_features,)

        if feature_counts is None:
            # shape: (num_features, 3) — для значений 0,1,2
            feature_counts = np.zeros((len(f), 3), dtype=int)

        # проверим, что все значения в {0,1,2}
        assert ((f >= 0) & (f <= 2)).all()

        for val in (0, 1, 2):
            feature_counts[:, val] += (f == val).astype(int)

# построение сгруппированной гистограммы
if feature_counts is not None:
    num_features = feature_counts.shape[0]

    # увеличим расстояние между признаками
    group_spacing = 1.5  # чем больше, тем больше отступ между признаками
    x = np.arange(num_features) * group_spacing

    width = 0.25  # ширина одного столбца

    plt.figure(figsize=(14, 6))
    plt.bar(x - width, feature_counts[:, 0], width=width, label='0')
    plt.bar(x,          feature_counts[:, 1], width=width, label='1')
    plt.bar(x + width, feature_counts[:, 2], width=width, label='2')

    plt.xticks(x, phonetic_features, rotation=90)
    plt.ylabel("Count")
    plt.title("Feature value counts (0/1/2) per feature")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
input_size = len(train_dataset[0][0])
num_features = len(train_dataset[0][-1])
train_dataset[0][-1]


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
# import torch.nn as nn
# 
# class FeatureClassificationModel(nn.Module):
#     def __init__(self, input_size, num_classes_per_head, dropout=0.3):
#         """
#         num_classes_per_head: список длинной num_features, 
#         например [3, 2, 3, 2, ...]
#         """
#         super().__init__()
#         
#         self.num_features = len(num_classes_per_head)
#         self.num_classes_per_head = num_classes_per_head
#         
#         self.linear1 = nn.Linear(input_size, 512)
#         self.linear2 = nn.Linear(512, 256)
#         self.linear3 = nn.Linear(256, 128)
#         self.linear4 = nn.Linear(128, 64)
#         
#         self.relu = nn.ReLU()
#         self.dropout = nn.Dropout(p=dropout)
#         
#         # === Multi-head classifier ===
#         self.heads = nn.ModuleList([
#             nn.Linear(64, n_cls) 
#             for n_cls in num_classes_per_head
#         ])
# 
#     def forward(self, x):
#         x = self.relu(self.linear1(x))
#         x = self.dropout(x)
#         
#         x = self.relu(self.linear2(x))
#         x = self.dropout(x)
#         
#         x = self.relu(self.linear3(x))
#         x = self.dropout(x)
#         
#         x = self.relu(self.linear4(x))
#         x = self.dropout(x)
#         
#         # Прогоняем каждую голову отдельно
#         outputs = [head(x) for head in self.heads]
#         
#         # outputs — это список тензоров [batch, n_classes_i]
#         # с разным числом классов, поэтому stack нельзя использовать
#         return outputs


In [ ]:
def compute_class_weights(dataset, num_features):

    all_targets = []

    for _, _, _, _, feats in dataset:
        all_targets.append(feats)

    all_targets = torch.stack(all_targets)  # (N, 24)

    weights_dict = {}

    for i in range(num_features):

        values = all_targets[:, i]
        counter = Counter(values.tolist())

        total = sum(counter.values())
        num_classes = len(counter)

        # формула inverse frequency
        weights = []

        for cls in sorted(counter.keys()):
            freq = counter[cls]
            weight = total / (num_classes * freq)
            weights.append(weight)

        weights_dict[i] = torch.tensor(weights, dtype=torch.float)

    return weights_dict

In [ ]:
compute_class_weights(train_dataset, num_features)

In [ ]:
FEATURE_NAMES = [
    'syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid',
    'voi', 'sg', 'cg', 'ant', 'cor', 'distr', 'lab',
    'hi', 'lo', 'back', 'round', 'velaric',
    'tense', 'long', "add", 'other'
]

NUM_FEATURES = len(FEATURE_NAMES)

In [ ]:
def infer_num_classes(dataset):

    all_features = []

    for _, _, _, _, feats in dataset:
        all_features.append(feats)

    all_features = torch.stack(all_features)

    num_classes = []

    for i in range(all_features.shape[1]):
        num_classes.append(len(torch.unique(all_features[:, i])))

    return num_classes

In [ ]:
infer_num_classes(train_dataset)

In [ ]:
class PhonemeFeaturePredictor(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        num_classes_per_feature=None,
        active_heads=None
    ):
        super().__init__()

        if num_classes_per_feature is None:
            num_classes_per_feature = [3] * NUM_FEATURES

        self.num_classes_per_feature = num_classes_per_feature

        # какие головы активны
        if active_heads is None:
            active_heads = list(range(NUM_FEATURES))
        self.active_heads = active_heads

        # общий backbone
        self.backbone = nn.Sequential(
            # nn.Linear(input_dim, 512),
            # nn.ReLU(),
            nn.Linear(input_dim, 128),
            nn.ReLU(),
        )

        # отдельная голова на каждый признак
        self.heads = nn.ModuleList([
            nn.Linear(128, num_classes_per_feature[i])
            for i in range(NUM_FEATURES)
        ])

    def forward(self, x):
        shared = self.backbone(x)

        outputs = []
        for i, head in enumerate(self.heads):
            if i in self.active_heads:
                outputs.append(head(shared))
            else:
                outputs.append(None)

        return outputs

In [ ]:
def compute_loss(outputs, targets, active_heads, criterion_dict):
    total_loss = 0
    losses_per_feature = {}

    for i in active_heads:
        logits = outputs[i]
        target = targets[:, i]

        # print(f"Feature {i}")
        # print("Num classes:", logits.shape[1])
        # print("Target min:", target.min().item())
        # print("Target max:", target.max().item())

        loss = criterion_dict[i](logits, target)

        total_loss += loss
        losses_per_feature[i] = loss.item()

    return total_loss, losses_per_feature

In [ ]:
def compute_accuracy(outputs, targets, active_heads):
    accuracies = {}

    for i in active_heads:

        logits = outputs[i]
        preds = torch.argmax(logits, dim=1)

        correct = (preds == targets[:, i]).float().mean().item()

        accuracies[i] = correct

    # общая accuracy — средняя по признакам
    overall = sum(accuracies.values()) / len(accuracies)

    return overall, accuracies

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    device,
    active_heads,
    criterion_dict,
    epochs=10,
    eval_steps_per_epoch=4,
    save_model_path = None
):

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_feature_acc": {i: [] for i in active_heads},
        "val_feature_acc": {i: [] for i in active_heads},
        "steps": []
    }
    best_test_loss = float('inf')

    step = 0
    eval_interval = len(train_loader) // eval_steps_per_epoch

    for epoch in range(epochs):

        model.train()

        running_loss = 0
        running_acc = 0
        per_feature_acc = {i: 0 for i in active_heads}

        for batch_idx, (embeddings, _, _, _, features) in enumerate(train_loader):

            embeddings = embeddings.to(device)
            targets = features.to(device)

            optimizer.zero_grad()

            outputs = model(embeddings)

            loss, _ = compute_loss(outputs, targets, active_heads, criterion_dict)

            loss.backward()
            optimizer.step()

            overall_acc, acc_dict = compute_accuracy(outputs, targets, active_heads)

            running_loss += loss.item()
            running_acc += overall_acc

            for i in active_heads:
                per_feature_acc[i] += acc_dict[i]

            step += 1

            # промежуточная оценка
            if (batch_idx + 1) % eval_interval == 0:

                train_loss = running_loss / (batch_idx + 1)
                train_acc = running_acc / (batch_idx + 1)

                train_feature_acc = {
                    i: per_feature_acc[i] / (batch_idx + 1)
                    for i in active_heads
                }

                val_loss, val_acc, val_feature_acc = evaluate(
                    model, val_loader, device, active_heads, criterion_dict
                )

                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)

                history["train_acc"].append(train_acc)
                history["val_acc"].append(val_acc)

                for i in active_heads:
                    history["train_feature_acc"][i].append(train_feature_acc[i])
                    history["val_feature_acc"][i].append(val_feature_acc[i])

                history["steps"].append(step)

                print(
                    f"Epoch {epoch+1} | step {batch_idx+1}/{len(train_loader)} | "
                    f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
                    f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f}"
                )
            scheduler.step()
            
        if val_loss < best_test_loss:
            best_test_loss = val_loss
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                    "train_loss": train_loss,
                    "test_loss": best_test_loss,
                },
                save_model_path,
            )
            print(f"  → Best model saved at epoch {epoch+1} with test_loss={best_test_loss:.4f}")
    

    return history

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, active_heads, criterion_dict):
    model.eval()

    total_loss = 0
    total_acc = 0
    per_feature_acc = {i: 0 for i in active_heads}

    for embeddings, _, _, _, features in loader:

        embeddings = embeddings.to(device)
        targets = features.to(device)

        outputs = model(embeddings)

        loss, _ = compute_loss(outputs, targets, active_heads, criterion_dict)

        overall_acc, acc_dict = compute_accuracy(outputs, targets, active_heads)

        total_loss += loss.item()
        total_acc += overall_acc

        for i in active_heads:
            per_feature_acc[i] += acc_dict[i]

    n = len(loader)

    return (
        total_loss / n,
        total_acc / n,
        {i: per_feature_acc[i] / n for i in active_heads}
    )

In [ ]:
def evaluate_before_training(model, loader, device, active_heads, criterion_dict):

    loss, acc, feature_acc = evaluate(
        model,
        loader,
        device,
        active_heads,
        criterion_dict
    )

    print("\nBaseline before training:")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")

    for k, v in feature_acc.items():
        print(f"Feature {k} acc: {v:.4f}")

    return loss, acc, feature_acc

In [ ]:
def plot_feature_accuracy(history, baseline_feature_acc, active_heads):

    steps = history["steps"]

    for i, head in enumerate(active_heads):
        if i in baseline_feature_acc.keys():
            
            
            plt.figure()
            
            plt.scatter(0, baseline_feature_acc[i], label='baseline')
    
            plt.plot(
                steps,
                history["train_feature_acc"][i],
                label="train"
            )
    
            plt.plot(
                steps,
                history["val_feature_acc"][i],
                label="val"
            )
    
            plt.title(f"Feature '{head}' accuracy")
            plt.xlabel("step")
            plt.ylabel("accuracy")
            plt.legend()
    
            plt.show()
        
import matplotlib.pyplot as plt


def plot_training(history, baseline_loss, baseline_acc):

    steps = history["steps"]

    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.scatter(0, baseline_loss, label='baseline')
    plt.plot(steps, history["train_loss"], label="train")
    plt.plot(steps, history["val_loss"], label="val")
    plt.title("Loss")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.legend()

    plt.subplot(1,2,2)
    plt.scatter(0, baseline_acc, label='baseline')
    plt.plot(steps, history["train_acc"], label="train")
    plt.plot(steps, history["val_acc"], label="val")
    plt.title("Accuracy")
    plt.xlabel("step")
    plt.ylabel("accuracy")
    plt.legend()

    plt.show()

In [ ]:
num_classes_per_feature = infer_num_classes(train_dataset)

In [ ]:
#num_classes_per_feature = [3] * NUM_FEATURES

In [ ]:
#active_heads = list(range(NUM_FEATURES))
active_heads = [FEATURE_NAMES.index(f) for f in ['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid',
                                                'voi', 'ant', 'cor', 'distr', 'lab',
                                                'hi', 'lo', 'back', 'round',
                                                'tense']]
active_heads_names = ['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid',
                                                'voi', 'ant', 'cor', 'distr', 'lab',
                                                'hi', 'lo', 'back', 'round',
                                                'tense']
                                                

In [ ]:
model = PhonemeFeaturePredictor(
    input_dim=256,
    hidden_dim=256,
    num_classes_per_feature=num_classes_per_feature,
    active_heads=active_heads
).to(device)
criterion_dict = {}

In [ ]:
for i in active_heads:
    criterion_dict[i] = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

In [ ]:
baseline_loss, baseline_acc, baseline_feature_acc = evaluate_before_training(
    model,
    test_loader,
    device,
    active_heads,
    criterion_dict
)

In [ ]:
# history = {
#     "train_loss": [], "test_loss": [],
#     "train_acc": [], "test_acc": [],
#     "train_feature_acc": {i: [] for i in active_heads},
#     "test_feature_acc": {i: [] for i in active_heads},
#     # новые поля для половин
#     "train_half_loss": [], "test_half_loss": [],
#     "train_half_acc": [], "test_half_acc": [],
#     "train_half_feature_acc": {i: [] for i in active_heads},
#     "test_half_feature_acc": {i: [] for i in active_heads},
# }

In [ ]:

history = train_model(
    model,
    train_loader,
    test_loader,
    optimizer,
    scheduler,
    device,
    active_heads,
    criterion_dict,
    epochs=5,
    eval_steps_per_epoch=4,
    save_model_path=save_model_path
)



In [ ]:
plot_training(history,  baseline_loss, baseline_acc)
plot_feature_accuracy(history,baseline_feature_acc, FEATURE_NAMES)

In [ ]:
import torch
from sklearn.metrics import confusion_matrix


@torch.no_grad()
def compute_confusion_matrices(model, loader, device, active_heads):

    model.eval()

    all_preds = {i: [] for i in active_heads}
    all_targets = {i: [] for i in active_heads}

    for embeddings, _, _, _, features in loader:

        embeddings = embeddings.to(device)
        targets = features.to(device)

        outputs = model(embeddings)

        for i in active_heads:

            logits = outputs[i]
            preds = torch.argmax(logits, dim=1)

            all_preds[i].extend(preds.cpu().numpy())
            all_targets[i].extend(targets[:, i].cpu().numpy())

    confusion_matrices = {}

    for i in active_heads:

        cm = confusion_matrix(
            all_targets[i],
            all_preds[i]
        )

        confusion_matrices[i] = cm

    return confusion_matrices

In [ ]:
import seaborn as sns
def plot_confusion_matrices(confusion_matrices, feature_names):
    num = len(confusion_matrices)
    rows = int(np.ceil(num / 3))
    cols = 3
    fig, axes = plt.subplots(ncols=cols, nrows=rows, figsize=(18, 5*rows))
    axes = axes.flatten()

    for i, cm in enumerate(confusion_matrices.items()):  # enumerate вместо range
        feature_idx, cm_data = cm  # распаковываем tuple
        
        ax = axes[i]  # i от 0, а не i-1

        sns.heatmap(
            cm_data,  # используем cm_data вместо cm
            annot=True,
            fmt="d",
            cmap="Blues",
            ax=ax,
            cbar_kws={'shrink': 0.8}  # компактная цветовая шкала
        )

        ax.set_title(f"Confusion Matrix: {feature_names[feature_idx]}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        
        # если матрица 3x3 (для значений 0/1/2), добавим подписи
        if cm_data.shape == (3, 3):
            ax.set_xticklabels(['0', '1', '2'])
            ax.set_yticklabels(['0', '1', '2'])

    # убираем пустые subplot'ы
    for i in range(num, len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_confusion_matrices_norm(confusion_matrices, feature_names):
    num = len(confusion_matrices)
    rows = int(np.ceil(num / 3))
    cols = 3
    fig, axes = plt.subplots(ncols=cols, nrows=rows, figsize=(18, 5*rows))
    axes = axes.flatten()

    for i, cm in enumerate(confusion_matrices.items()):
        feature_idx, cm_data = cm
        
        ax = axes[i]

        # НОРМИРОВКА ПО СТРОКАМ (проценты предсказаний для каждого истинного класса)
        cm_rel = cm_data.astype(float) / cm_data.sum(axis=1, keepdims=True) * 100
        
        sns.heatmap(
            cm_rel,  # относительные значения
            annot=True,
            fmt=".1f",      # 1 знак после запятой
            cmap="Blues",
            ax=ax,
            cbar_kws={'label': '%', 'format': '%.0f%%'},  # цветовая шкала в %
            xticklabels=['0', '1', '2'],
            yticklabels=['0', '1', '2'],
            vmin=0, vmax=100
        )

        ax.set_title(f"Confusion Matrix: {feature_names[feature_idx]}\n(Row %)")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")

    for i in range(num, len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()


In [ ]:
confusion_matrices = compute_confusion_matrices(model, test_loader, device, active_heads)


In [ ]:
plot_confusion_matrices(confusion_matrices, FEATURE_NAMES)

In [ ]:
plot_confusion_matrices_norm(confusion_matrices, FEATURE_NAMES)

## Inference

In [ ]:
checkpoint = torch.load(save_model_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
if checkpoint["scheduler_state_dict"] is not None:
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

In [ ]:
model.eval()

        

In [ ]:
for embeddings, _, _, ipa_symbols, features in test_dataset:
    
    if ipa_symbols not in [['k'], ['a']]:
        embeddings = embeddings.to(device)
        targets = features.to(device)
        print(ipa_symbols)

        outputs = model(embeddings)
        for i in active_heads:

            logits = outputs[i]
            preds = torch.argmax(logits, dim=0)
            print(phonetic_features[i],targets[i].cpu().numpy(), preds.cpu().numpy())
        break
    else:
        continue

In [ ]:
from collections import defaultdict
symbol_features = []
symbols_checked = []
true_classes = []
# stats[symbol][head][pred_class] = count
stats = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

model.eval()
with torch.no_grad():
    for embeddings, _, _, ipa_symbols, features in test_dataset:
        embeddings = embeddings.to(device)
        outputs = model(embeddings)  
        
        symbol = ipa_symbols[0]          # или ipa_symbols[0], см. свою структуру

        for i in active_heads:
            logits = outputs[i]       # размер [num_classes_i] или [1, num_classes_i]
            if logits.dim() > 1:
                logits = logits.squeeze(0)

            pred = torch.argmax(logits, dim=-1).item()   # предсказанный класс (int)

            stats[symbol][i][pred] += 1
            true_classes.append({'symbol':ipa_symbols, "head":i, 'true_class':features[i]})
            


In [ ]:
symbol_features = []

for symbol, heads_dict in stats.items():
    for head_id, class_counts in heads_dict.items():
        symbol_features.append({
            "symbol": symbol,
            "head": head_id,
            "counts": dict(class_counts),  # {класс: сколько раз предсказан}
        })


In [ ]:
symbol_features

In [ ]:
true_classes

In [ ]:
# создаём уникальный ключ из трёх полей
unique_true_classes = list(
    dict.fromkeys(
        (tuple(d['symbol']), d['head'], d['true_class'].item()) 
        for d in true_classes
    )
)

# преобразуем обратно в словари
unique_true_classes = [
    {'symbol':list(key[0])[0], 'head': key[1], 'true_class': key[2]}
    for key in unique_true_classes
]

In [ ]:
unique_true_classes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = symbol_features
true_data = unique_true_classes

# превращаем counts в "длинную" таблицу
rows = []
for item in data:
    symbol = item['symbol']
    head = item['head']
    for cls, count in item['counts'].items():
        rows.append({
            'symbol': symbol,
            'head': head,
            'class': cls,
            'count': count
        })
df = pd.DataFrame(rows)
true_df = pd.DataFrame(true_data)

# строим графики
for symbol, subdf in df.groupby('symbol'):
    plt.figure(figsize=(12,6))
    
    ax = sns.barplot(
        data=subdf,
        x='head',
        y='count',
        hue='class'
    )
    
    # добавляем красные точки для true class
    true_sub = true_df[true_df['symbol'] == symbol]
    
    for head, true_cls in zip(true_sub['head'], true_sub['true_class']):
        # ищем прямоугольник с нужной head и class
        for rect in ax.patches:
            rect_head = rect.get_x() + rect.get_width()/2  # центр столбца
            rect_class = rect.get_facecolor()  # не используем, просто чтобы иметь patch
            # проверяем, что это столбец с нашим head и классом
            if rect.get_height() > 0 and int(subdf.iloc[0]['head']) == head:  # заглушка
                pass  # не используем для позиционирования
        # проще: рисуем точку над центром head
        x_pos = head  # здесь matplotlib сам располагает категории как числа
        ax.scatter(x_pos, 0, color='red', s=120, zorder=10)

    plt.title(f'Prediction distribution for symbol "{symbol}"')
    plt.xlabel('Head')
    plt.ylabel('Count')
    plt.legend(title='Predicted class')
    plt.tight_layout()
    plt.show()
    

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = symbol_features

# превращаем в "длинную" таблицу
rows = []
for item in data:
    symbol = item['symbol']
    head = item['head']
    for cls, count in item['counts'].items():
        rows.append({
            'symbol': symbol,
            'head': head,
            'class': cls,
            'count': count
        })

df = pd.DataFrame(rows)

# строим графики для каждого символа
for symbol, subdf in df.groupby('symbol'):
    plt.figure(figsize=(10,5))
    
    sns.barplot(
        data=subdf,
        x='head',
        y='count',
        hue='class'
    )
    
    plt.title(f'Prediction distribution for symbol "{symbol}"')
    plt.xlabel('Head')
    plt.ylabel('Count')
    plt.legend(title='Predicted class')
    plt.tight_layout()
    plt.show()